# 1. Machine Learning Basics

**AI and Economics Summer School** — Andrea Ciccarone

Everything here runs on CPU with only `numpy`, `scikit-learn` and `matplotlib`.

What we do:
1. Watch a model overfit
2. Split the data properly
3. Cross-validate to pick a penalty
4. Compare ridge, lasso, a forest and a boosted model
5. Evaluate a classifier with AUC rather than accuracy

The point is not the models. It is the **evaluation discipline** that makes any
of them trustworthy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

rng = np.random.default_rng(0)
plt.rcParams["figure.dpi"] = 120

## 1. Overfitting

We simulate data from a known function, then fit polynomials of increasing degree.
Because we know the truth, we can see exactly how wrong each fit is.

In [ ]:
def truth(x):
    return np.sin(2 * np.pi * x)

n = 30
x = np.sort(rng.uniform(0, 1, n))
y = truth(x) + rng.normal(0, 0.28, n)
grid = np.linspace(0, 1, 400)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, deg in zip(axes, [1, 4, 15]):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(x[:, None], y)
    train_mse = np.mean((y - model.predict(x[:, None])) ** 2)
    ax.plot(grid, truth(grid), "--", color="gray", label="truth")
    ax.scatter(x, y, s=20, color="#0C2244", zorder=3)
    ax.plot(grid, model.predict(grid[:, None]), color="#964F4C", lw=2)
    ax.set_title(f"degree {deg} | train MSE {train_mse:.3f}")
    ax.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

The degree-15 fit has the **lowest training error and is the worst model**.

This is why training error is never evidence. Hold data out.

## 2. Train / test, and why the split must respect structure

The mechanical split is one line. The judgement is in *how* you split.

In [ ]:
X = x[:, None]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

for deg in [1, 4, 15]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(X_tr, y_tr)
    tr = np.mean((y_tr - m.predict(X_tr)) ** 2)
    te = np.mean((y_te - m.predict(X_te)) ** 2)
    print(f"degree {deg:2d} | train {tr:.3f} | test {te:.3f}")

> **Warning for economists.** `train_test_split` assumes observations are exchangeable.
> If you have repeated observations of the same firm, judge or channel, use
> `GroupKFold` so all rows of a group land in the same fold. With time series use
> `TimeSeriesSplit`. Otherwise the model sees near-copies of the test data and your
> out-of-sample number is fiction.

## 3. Cross-validation to choose the penalty

Ridge and lasso both need a $\lambda$. Nothing in economics tells us what it is,
so we let held-out data decide.

In [ ]:
# A higher-dimensional problem so regularization actually matters.
n, p = 120, 60
Xb = rng.normal(size=(n, p))
beta_true = np.zeros(p)
beta_true[:5] = [3.0, -2.0, 1.5, 2.5, -1.0]   # only 5 of 60 regressors matter
yb = Xb @ beta_true + rng.normal(0, 1.0, n)

alphas = np.logspace(-3, 2, 40)
ridge_cv = GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                        {"ridge__alpha": alphas},
                        cv=KFold(5, shuffle=True, random_state=0),
                        scoring="neg_mean_squared_error").fit(Xb, yb)
lasso_cv = GridSearchCV(make_pipeline(StandardScaler(), Lasso(max_iter=20000)),
                        {"lasso__alpha": alphas},
                        cv=KFold(5, shuffle=True, random_state=0),
                        scoring="neg_mean_squared_error").fit(Xb, yb)

print("best ridge alpha:", round(ridge_cv.best_params_["ridge__alpha"], 4))
print("best lasso alpha:", round(lasso_cv.best_params_["lasso__alpha"], 4))

lasso_coef = lasso_cv.best_estimator_.named_steps["lasso"].coef_
print("lasso selected", int((np.abs(lasso_coef) > 1e-8).sum()), "of", p, "regressors")
print("truly nonzero:", int((beta_true != 0).sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.stem(beta_true, linefmt="C7-", markerfmt="C7o", basefmt=" ", label="true beta")
ax.stem(lasso_coef, linefmt="-", markerfmt="o", basefmt=" ", label="lasso estimate")
ax.set_xlabel("regressor index"); ax.legend(frameon=False)
ax.set_title("Lasso finds the big coefficients, plus a tail of false positives")
plt.tight_layout(); plt.show()

Notice the count printed above: lasso keeps far more regressors than the five that
truly matter. That is not a bug. The $\lambda$ was chosen to minimise **prediction
error**, and for prediction it is cheap to carry a few small spurious coefficients.

So: lasso is a prediction device, not a variable-selection oracle. Do not read a
lasso-selected set as "the variables that matter".

**Exercise.** Make two of the true regressors highly correlated (e.g. set
`Xb[:, 1] = Xb[:, 0] + 0.1 * rng.normal(size=n)`) and re-run with a few different
seeds. Which of the two does lasso keep?

## 4. Classification: forests, boosting, and honest evaluation

Now a binary outcome, which is the shape almost every text or image measurement task takes.

In [ ]:
from sklearn.datasets import make_classification

Xc, yc = make_classification(n_samples=2000, n_features=20, n_informative=5,
                             n_redundant=5, weights=[0.85, 0.15],  # deliberately unbalanced
                             random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3,
                                              stratify=yc, random_state=1)
print("share of positives:", yc.mean().round(3))

In [ ]:
models = {
    "Logistic":       make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest":  RandomForestClassifier(n_estimators=400, random_state=0),
    "Gradient boost": GradientBoostingClassifier(random_state=0),
}

fig, ax = plt.subplots(figsize=(5.5, 4))
for name, m in models.items():
    m.fit(Xc_tr, yc_tr)
    p = m.predict_proba(Xc_te)[:, 1]
    acc = (m.predict(Xc_te) == yc_te).mean()
    auc = roc_auc_score(yc_te, p)
    print(f"{name:16s} accuracy {acc:.3f} | AUC {auc:.3f}")
    fpr, tpr, _ = roc_curve(yc_te, p)
    ax.plot(fpr, tpr, lw=2, label=f"{name} ({auc:.3f})")

ax.plot([0, 1], [0, 1], "--", color="lightgray")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

print("\nAlways-predict-zero accuracy:", round(1 - yc_te.mean(), 3))

Note the last line. A model that predicts zero for everyone gets high accuracy here
and has **no information whatsoever**. Its AUC would be 0.5.

With unbalanced classes — which is the normal case when you are detecting something
in text or images — report AUC, or precision and recall at a stated threshold.

In [ ]:
best = models["Gradient boost"]
pred = best.predict(Xc_te)
print(confusion_matrix(yc_te, pred))
print()
print(classification_report(yc_te, pred, digits=3))

## Exercises

1. Change the class balance to `weights=[0.98, 0.02]`. What happens to accuracy?
   To AUC? Which one told you the truth?
2. Fit the forest with `max_depth=2`. Does test AUC go up or down? Why?
3. Take the polynomial example from section 1 and use `cross_val_score` to pick
   the degree automatically. Does it choose 4?